In [ ]:
!pip install opencv-python scikit-image scikit-learn tqdm

In [12]:
import os
import cv2
import numpy as np
from tqdm import tqdm
from skimage.feature import local_binary_pattern
from sklearn.svm import SVC
from sklearn.pipeline import Pipeline
from sklearn.preprocessing import StandardScaler
from sklearn.metrics import classification_report

# ==========================
# CONFIG
# ==========================

DATA_ROOT = r"e:\PythonFile\Project\Facial-Recognition\data\CASIA_faceAntisp"

IMG_SIZE = 64
P = 8
R = 1
METHOD = "uniform"
COLOR_SPACE = "gray" 

FPS = 25
WINDOW_SEC = 3
OVERLAP_SEC = 2

REAL = 1
ATTACK = 0


# ==========================
# LBP FEATURE
# ==========================

def extract_lbp(gray):
    lbp = local_binary_pattern(gray, P, R, method=METHOD)

    if METHOD == "uniform":
        n_bins = P + 2
    else:
        n_bins = 2 ** P

    hist, _ = np.histogram(
        lbp.ravel(),
        bins=n_bins,
        range=(0, n_bins)
    )

    hist = hist.astype("float")
    hist /= (hist.sum() + 1e-6)

    return hist


# ==========================
# VIDEO → FRAME FEATURES
# ==========================

def convert_color(frame):
    if COLOR_SPACE == "gray":
        img = cv2.cvtColor(frame, cv2.COLOR_BGR2GRAY)
        img = cv2.resize(img, (IMG_SIZE, IMG_SIZE))
        return [img]

    elif COLOR_SPACE == "rgb":
        img = cv2.cvtColor(frame, cv2.COLOR_BGR2RGB)
        img = cv2.resize(img, (IMG_SIZE, IMG_SIZE))
        return cv2.split(img)

    elif COLOR_SPACE == "hsv":
        img = cv2.cvtColor(frame, cv2.COLOR_BGR2HSV)
        img = cv2.resize(img, (IMG_SIZE, IMG_SIZE))
        return cv2.split(img)

    elif COLOR_SPACE == "ycrcb":
        img = cv2.cvtColor(frame, cv2.COLOR_BGR2YCrCb)
        img = cv2.resize(img, (IMG_SIZE, IMG_SIZE))
        return cv2.split(img)

def extract_video_features(video_path):
    cap = cv2.VideoCapture(video_path)
    features = []

    while True:
        ret, frame = cap.read()
        if not ret:
            break

        channels = convert_color(frame)

        channel_features = []
        for ch in channels:
            hist = extract_lbp(ch)
            channel_features.extend(hist)

        features.append(channel_features)

    cap.release()
    return np.array(features)


# ==========================
# TEMPORAL WINDOW
# ==========================

def temporal_windows(features):
    window_size = int(FPS * WINDOW_SEC)
    overlap = int(FPS * OVERLAP_SEC)
    step = window_size - overlap

    windows = []
    for start in range(0, len(features) - window_size + 1, step):
        window = features[start:start+window_size]
        windows.append(np.mean(window, axis=0))

    return windows


def first_window(features):
    window_size = int(FPS * WINDOW_SEC)

    if len(features) < window_size:
        return None

    return np.mean(features[:window_size], axis=0)


# ==========================
# LABEL FUNCTION
# ==========================

def get_label(filename):
    name = filename.lower()

    if name.startswith("1") or name.startswith("2") or \
       name.startswith("HR_1"):
        return REAL
    else:
        return ATTACK


# ==========================
# LOAD DATA
# ==========================

def load_split(split="train"):
    root = os.path.join(DATA_ROOT, f"{split}_release")

    X = []
    y = []

    subjects = os.listdir(root)

    for subject in tqdm(subjects):
        subject_path = os.path.join(root, subject)

        for video in os.listdir(subject_path):
            video_path = os.path.join(subject_path, video)

            label = get_label(video)

            frame_feats = extract_video_features(video_path)

            if split == "train":
                windows = temporal_windows(frame_feats)
                for w in windows:
                    X.append(w)
                    y.append(label)
            else:
                feat = first_window(frame_feats)
                if feat is not None:
                    X.append(feat)
                    y.append(label)

    return np.array(X), np.array(y)


# ==========================
# TRAIN + TEST
# ==========================

print("Extracting training data...")
X_train, y_train = load_split("train")

print("Extracting testing data...")
X_test, y_test = load_split("test")
# ==========================
# SAVE FEATURES
# ==========================

np.savez("E:/PythonFile/Project/Facial-Recognition/data/feature/casia_lbp_u_gray_features.npz",
         X_train=X_train,
         y_train=y_train,
         X_test=X_test,
         y_test=y_test)

print("Saved features to casia_lbp_features.npz")


Extracting training data...


100%|██████████| 20/20 [01:37<00:00,  4.85s/it]


Extracting testing data...


100%|██████████| 30/30 [01:36<00:00,  3.21s/it]

Saved features to casia_lbp_features.npz
